# 03 · MCP 함수 직접 호출로 데이터 가져오기

**파이프라인 3/4** — 에이전트/LLM 없이 **MCP 툴을 직접 호출**해 원시 데이터를 가져옵니다.
각 서버가 어떤 툴을 제공하고 무엇을 반환하는지 감을 잡는 단계입니다. (pipeline 패키지 없이 독립 실행)

## 셋업 (설정 + 인증 + MCP 헬퍼)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')
TOKEN_CACHE = PROJECT_ROOT / '.token_cache.json'   # 로그인 1회 → 이후 노트북/웹앱이 재사용

# 2) 설정값 (.env에서 읽음)
TENANT_ID     = os.environ.get('TENANT_ID') or '00000000-0000-0000-0000-000000000000'
CLIENT_ID     = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')
AUTH_MODE     = os.environ.get('AUTH_MODE', 'device_code')   # device_code | client_credentials
AUTHORITY     = f'https://login.microsoftonline.com/{TENANT_ID}'

def _mcp_url(env_var, server_id):
    return (os.environ.get(env_var) or '').strip() or \
        f'https://agent365.svc.cloud.microsoft/agents/tenants/{TENANT_ID}/servers/{server_id}'

# 두 MCP 소스: Mail / Teams. 서버마다 delegated 스코프가 달라 토큰도 소스별로 1개씩.
SOURCES = {
    'mail':  {'label': 'Mail',  'url': _mcp_url('MAIL_MCP_SERVER_URL',  'mcp_MailTools')},
    'teams': {'label': 'Teams', 'url': _mcp_url('TEAMS_MCP_SERVER_URL', 'mcp_TeamsServer')},
}
for _s in SOURCES.values():
    _s['scopes'] = [f"{_s['url']}/.default"]

# 3) MSAL 인증 — 소스별 access token. device_code는 최초 1회만 로그인, 이후 캐시 재사용.
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token(source_key: str) -> str:
    """소스 1개의 access token 확보 (없으면 device-code 로그인)."""
    assert CLIENT_ID, 'CLIENT_ID가 .env에 없습니다 (.env.example 참고).'
    scopes = SOURCES[source_key]['scopes']
    if AUTH_MODE == 'client_credentials':           # 앱 전용 토큰
        app = msal.ConfidentialClientApplication(
            CLIENT_ID, authority=AUTHORITY, client_credential=CLIENT_SECRET, token_cache=_cache)
        res = app.acquire_token_for_client(scopes=scopes)
    else:                                           # 위임(사용자) 로그인
        app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
        accounts = app.get_accounts()
        res = app.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
        if not (res and 'access_token' in res):     # 캐시에 없으면 device-code 로그인
            flow = app.initiate_device_flow(scopes=scopes)
            print(flow['message'])                   # https://microsoft.com/devicelogin + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TENANT_ID   :', TENANT_ID, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
for _k, _s in SOURCES.items():
    print(f"  [{_k}] {_s['label']} -> {_s['url']}")

In [ ]:
# --- MCP 읽기 헬퍼: 연결은 호출할 때마다 열고 항상 닫는다 ---
async def _with_session(source_key, token, fn):
    url = SOURCES[source_key]['url']
    async with streamablehttp_client(url, headers={'Authorization': f'Bearer {token}'}) as (r, w, _):
        async with ClientSession(r, w) as s:      # MCP 세션 초기화(handshake)
            await s.initialize()
            return await fn(s)

async def list_tools(token, source_key):
    """MCP 서버가 노출하는 도구(함수) 목록."""
    res = await _with_session(source_key, token, lambda s: s.list_tools())
    return list(res.tools or [])

async def call_tool(token, source_key, name, args=None):
    """MCP 도구를 이름+인자로 직접 호출."""
    return await _with_session(source_key, token, lambda s: s.call_tool(name, arguments=args or {}))

def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def content_to_text(result) -> str:
    """MCP 도구 결과의 content 블록들을 읽기 좋은 문자열로 평탄화."""
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            parts.append(getattr(b, 'text', ''))
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

## 1. 토큰 + 툴 목록

In [ ]:
tokens = {}
for key in SOURCES:
    try:
        tokens[key] = ensure_token(key)
    except Exception as exc:
        print(f'{key} 토큰 실패: {exc}')

tools_by_source = {}
for key, token in tokens.items():
    tools_by_source[key] = await list_tools(token, key)
    print(f'{key}: ' + ', '.join(t.name for t in tools_by_source[key]))

## 2. 검색/조회 툴 후보 찾기
이름에 search/list/get/find/message/mail 등이 포함된 읽기 툴을 추려봅니다.

In [ ]:
import re
READ_HINT = re.compile(r'(search|list|get|find|read|message|mail|chat|channel)', re.I)
for key, ts in tools_by_source.items():
    print(f'\n=== {key} 읽기 툴 후보 ===')
    for t in ts:
        if READ_HINT.search(t.name):
            print(' -', t.name)

## 3. 툴 직접 호출
아래 `SOURCE`, `TOOL_NAME`을 위에서 확인한 실제 툴로 채우고 실행하면 입력 스키마를 출력합니다.

In [ ]:
SOURCE    = 'mail'      # 'mail' 또는 'teams'
TOOL_NAME = ''          # 위 후보 목록에서 실제 툴 이름으로 채우세요

if TOOL_NAME:
    tool = next((t for t in tools_by_source[SOURCE] if t.name == TOOL_NAME), None)
    assert tool, f'{TOOL_NAME} 을(를) {SOURCE} 툴 목록에서 찾을 수 없습니다.'
    print('입력 스키마:')
    print(json.dumps(tool_schema(tool), ensure_ascii=False, indent=2))
else:
    print('TOOL_NAME을 채운 뒤 다시 실행하세요.')

In [ ]:
# 스키마를 참고해 인자를 채우세요. 예) {'query': 'Postgres', 'top': 10}
ARGS = {}
if TOOL_NAME:
    result = await call_tool(tokens[SOURCE], SOURCE, TOOL_NAME, ARGS)
    print(content_to_text(result)[:4000])

✅ 원하는 데이터를 직접 가져올 수 있음을 확인했습니다. 다음은 이 과정을 자연어로
자동화하는 **04_nl_aggregate_to_md** 입니다.